In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload;
# To disable autoreload; run %autoreload 0

In [0]:
# %pip install "psycopg[pool]>=3.1.0"
# %pip install --upgrade "databricks-sdk>=0.133.0"
dbutils.library.restartPython()

In [0]:
import os
from pprint import pprint
os.environ["PGHOST"] = "ep-polished-silence-d8t2cf3b.database.us-east-2.cloud.databricks.com"
os.environ["LAKEBASE_ENDPOINT"] = "projects/weather-intelligence/branches/production/endpoints/primary"
os.environ["PGUSER"] ="tuvu.uwyo@gmail.com"

### destination_api.py

##### 1. Import DestinationAPI class and client connection

In [0]:
os.environ["GEOAPIFY_API_KEY"] = '649d5b3a92ed4bbfa97f8c40906640f4'
from destination_api import DestinationAPI
client_geo = DestinationAPI(os.environ["GEOAPIFY_API_KEY"])

##### 2. Test get_places_radius()

In [0]:
features = client_geo.get_places_radius(location = "Seattle, WA"
                                        ,radius_m=3000
                                        ,categories="entertainment.museum"
                                        ,limit=3
                                    )
pprint(features)

In [0]:
features[0].keys()

##### 3. Test get_place_details()

In [0]:
place_id = features[0]["properties"]["place_id"]
place_id

In [0]:
detail = client_geo.get_place_details(place_id)
pprint(detail)

In [0]:
len(detail)
detail.keys()

### destionation.py

##### 1. Test categorize()

In [0]:
from destination import categorize

In [0]:
categories = ["entertainment", "entertainment.museum", "building", "building.tourism"]

result = categorize(categories)
print(result)

In [0]:
matches = [
    CATEGORY_MAP[c]
    for c in sorted(categories, key=lambda c: c.count("."), reverse=True)
    if c in CATEGORY_MAP
]
matches

In [0]:
matches[0]

##### 2. Test normalize_place()

In [0]:
from destination import normalize_place

doc = normalize_place(feature = features[0], detail = detail, location = "Seattle, WA")
pprint(doc)

##### 3. Test fetch_destionations()

In [0]:
from destination import fetch_destinations

docs = fetch_destinations(
    "Laramie, WY"
    ,api_key=os.environ["GEOAPIFY_API_KEY"]
    ,limit=5
    ,fetch_details=False
)

print(len(docs))
pprint(docs[0])

##### 4. Test upsert_documents()

In [0]:
from destination import upsert_documents

In [0]:
count = upsert_documents(docs)
print(count)

### embeddings.py

##### 1. Test embed_unembedded_documents()

In [0]:
# %pip install sentence-transformers


In [0]:
from embeddings import embed_unembedded_documents

In [0]:
import lakebase
count_embedded = embed_unembedded_documents(    
    lakebase.get_connection
    ,documents_table="destination_documents"
    ,embeddings_table="destination_embeddings"
)
print(count_embedded)

### search.py

##### 1. Test search_destination_documents()

In [0]:
from search import search_destination_documents

In [0]:
results = search_destination_documents("outdoor viewpoint with a view", lakebase.get_connection, top_k=3)
pprint(results)

### routing.py

In [0]:
os.environ["ORS_API_KEY"]='eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjUyYjgyNjE2YTVlZjQ3MGE5MDc3MDNlYTkwYTA4MWIxIiwiaCI6Im11cm11cjY0In0='
from routing import RoutingAPI, RoutingAPIError
client_ors = RoutingAPI(
    api_key=os.environ["ORS_API_KEY"]
)

##### 1. Test get_travel_time()

In [0]:
result = client_ors.get_travel_time(
    origin="Waikiki Beach, Honolulu, HI"
    ,destination="Diamond Head, Honolulu, HI"
)
pprint(result, sort_dicts=False)

##### 2. Test get_travel_time_matrix()

In [0]:
result = client_ors.get_travel_time_matrix(
    locations=[
        "Waikiki Beach, Honolulu, HI"
        ,"Diamond Head, Honolulu, HI"
        ,"Ala Moana Center, Honolulu, HI"
    ]
    ,mode="driving"
)
pprint(result, sort_dicts=False)

# Decoded for your 3 Honolulu spots (index 0 = Waikiki, 1 = Diamond Head, 2 = Ala Moana), driving:

# from → to	distance	duration
# Waikiki → Diamond Head	2.33 km	4.9 min
# Waikiki → Ala Moana	3.55 km	8.6 min
# Diamond Head → Waikiki	3.45 km	7.3 min
# Diamond Head → Ala Moana	6.01 km	12.1 min
# Ala Moana → Waikiki	3.17 km	7.7 min
# Ala Moana → Diamond Head	5.50 km	12.5 min

In [0]:
try:
    client_ors.get_travel_time(
        origin="Waikiki Beach, Honolulu, HI"
        ,destination="Diamond Head, Honolulu, HI"
        ,mode="teleport"
    )
except ValueError as e:
    print("Correctly rejected invalid mode:", e)

In [0]:
try:
    client_ors.get_travel_time(
        origin="asdkfjhaskdjfh nowhere land"
        ,destination="Diamond Head, Honolulu, HI"
    )
except ValueError as e:
    print("Correctly failed to resolve location:", e)

### itinerary.py

In [0]:
import lakebase
from itinerary import (
    create_itinerary
    ,add_itinerary_items
    ,get_itinerary
    ,list_itineraries
    ,update_itinerary_status
    ,_item_row
)

TEST_USER = "tuvu.uwyo@gmail.com"

In [0]:
destinations = lakebase.run_query(
    """
    SELECT id, name FROM destination_documents 
    WHERE location ILIKE %s 
    and name != 'Unnamed'
    and category != 'landmark'
    LIMIT 4
    """
    ,("%New York City, NY%",)
)

pprint(destinations)

##### 0. Print items before execute many

In [0]:
items = [
    {
        "day_number": 1
        ,"sequence_order": 1
        ,"item_type": "attraction"
        ,"attraction_id": destinations[0]["id"]
        ,"title": destinations[0]["name"]
        ,"notes": "Morning 1st activitiy"
    }
    ,{
        "day_number": 1
        ,"sequence_order": 2
        ,"item_type": "attraction"
        ,"attraction_id": destinations[1]["id"]
        ,"title": destinations[1]["name"]
        ,"notes": "Morning 2nd activitiy"
    }
]

items

In [0]:
COLUMNS = [
    "id", "itinerary_id", "day_number", "sequence_order", "item_type",
    "attraction_id", "title", "start_time", "end_time",
    "latitude", "longitude", "weather_context", "notes"
]

preview_itinerary_id = "PREVIEW-not-the-real-id"  # real one is generated inside create_itinerary

for _, item in enumerate(items):
    row = _item_row(preview_itinerary_id, item)
    print(row)

##### 1. Test create_itinerary()

In [0]:
TEST_USER = "tuvu.uwyo@gmail.com"

itinerary = create_itinerary(
    user_id=TEST_USER
    ,destination="New York City, NY"
    ,start_date="2026-12-10"
    ,end_date="2026-12-12"
    ,items = items
    ,preferences={"interests": [ "non-landmark"]}
)
pprint(itinerary)
print(itinerary["id"])
print(len(itinerary['items']), len(items))

In [0]:
lakebase.run_write("TRUNCATE itineraries CASCADE")

##### 2. Test add_itinerary_items

In [0]:
count = add_itinerary_items(
    itinerary["id"]
    ,[
        {
            "day_number": 1
            ,"sequence_order": 1
            ,"item_type": "attraction"
            ,"attraction_id": destinations[0]["id"]
            ,"title": destinations[0]["name"]
            ,"notes": "Real attraction_id, exercises the FK"
        }
        ,{
            "day_number": 1
            ,"sequence_order": 2
            ,"item_type": "meal"
            ,"title": "Lunch break"
            ,"notes": "No attraction_id — FK column is nullable"
        }
    ]
)
print("Items inserted:", count)

##### 3. Test get_itinerary()

In [0]:
full = get_itinerary(itinerary["id"])
pprint(full)

In [0]:
len(full["items"])
full["items"][0]["sequence_order"]

##### 4. Test list_itineraries()

In [0]:
mine = list_itineraries(TEST_USER)
pprint(mine)

##### 5. Test update_itinerary_status()

In [0]:
updated = update_itinerary_status(itinerary["id"], "confirmed")
pprint( updated)

##### 6. Test errors handling

In [0]:
try:
    update_itinerary_status(itinerary["id"], "not_a_real_status")
except ValueError as e:
    print("Correctly rejected invalid status:", e)

result = get_itinerary("00000000-0000-0000-0000-000000000000")
print("Nonexistent itinerary correctly returns:", result)

In [0]:
try:
    add_itinerary_items(
        itinerary['id']
        ,[{
            "day_number": 2
            ,"sequence_order": 1
            ,"item_type": "attraction"
            ,"attraction_id": "this-id-does-not-exist"
            ,"title": "Should fail on the FK"
        }]
    )
except Exception as e:
    print("Correctly rejected bad attraction_id (FK violation):", type(e).__name__, e)

In [0]:
destinations[2]["id"]

In [0]:
try:
    add_itinerary_items(
        itinerary['id']
        ,[
            {
                "day_number": 2
                ,"sequence_order": 1
                ,"item_type": "attraction"
                ,"attraction_id": destinations[2]["id"]
                ,"title": destinations[2]["name"]
            }
            ,{
                "day_number": 2
                ,"sequence_order": 2
                ,"item_type": "attraction"
                ,"attraction_id": "this-id-does-not-exist"
                ,"title": "This one should fail"
            }
        ]
    )
except Exception as e:
    print("Batch correctly rejected:", type(e).__name__, e)

In [0]:
# Batch correctly rejected: ForeignKeyViolation insert or update on table "itinerary_items" violates foreign key constraint "itinerary_items_attraction_id_fkey"
# DETAIL:  Key (attraction_id)=(this-id-does-not-exist) is not present in table "destination_documents".

In [0]:
# The real test: did "This one is fine" get orphaned into the table
# even though its sibling in the same batch failed?
items = get_itinerary(itinerary['id'])["items"]
len(items) == 2, f"Expected 2, got {len(items)} — the good item leaked in despite the batch failing"

In [0]:

import sys
sys.version_info
